# Sector Summaries for All Periods

This notebook generates:
1. **Sector Imputation Summary**: Shows the percentage of stocks with imputed scope emissions by sector
2. **Sector Emissions Summary**: Shows emission statistics by sector

Both summaries are created for each period and then aggregated across all periods.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [2]:
# Define periods to analyze
PERIODS = ["0321", "0621", "0921", "1221", "0322", "0622", "0922", "1222", "0323", "0623", "0923", "1223"]

# Data directory
DATA_DIR = Path("data")
DATASETS_DIR = DATA_DIR / "datasets"

# Display options
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Helper Functions

In [3]:
def create_sector_imputation_summary(df, period):
    """
    Create sector imputation summary for a single period.
    Shows the percentage of stocks with imputed scope emissions by sector.
    
    Note: Excludes manual imputation columns (only uses automatic imputation flags)
    """
    auto_impute_flags = ['Scope 1 Imputed', 'Scope 2 Imputed', 'Scope 3 Imputed']
    filled_count_cols = ['Filled Scope 1 Count', 'Filled Scope 2 Count', 'Filled Scope 3 Count']
    
    summary_rows = []
    
    for sector, group in df.groupby('GICS Sector'):
        n_stocks = len(group)
        row = {
            'Period': period,
            'Sector': sector,
            'Number of Stocks': n_stocks
        }
        
        # Automatically imputed (Scope 1/2/3)
        for col in auto_impute_flags:
            count = group[col].fillna(0).sum()  # Treat NaN as 0 (not imputed)
            scope = col.split()[1]  # e.g. "1" from "Scope 1 Imputed"
            row[f'Imputed Scope {scope} Count'] = int(count)
            row[f'Imputed Scope {scope} %'] = count / n_stocks * 100
        
        # Add filled counts - now summing binary indicators (0 or 1) per sector
        for col in filled_count_cols:
            if col in group.columns:
                # Sum the binary indicators to get count of filled companies in this sector
                filled_count = group[col].fillna(0).sum()
                scope = col.split()[2]  # e.g. "1" from "Filled Scope 1 Count"
                row[f'Filled Scope {scope} Count'] = int(filled_count)
                row[f'Filled Scope {scope} %'] = (filled_count / n_stocks * 100) if n_stocks > 0 else 0
        
        summary_rows.append(row)
    
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.sort_values("Sector").reset_index(drop=True)
    
    # Round all percentage columns to integers
    percentage_cols = [col for col in summary_df.columns if '%' in col]
    summary_df[percentage_cols] = summary_df[percentage_cols].round(0).astype('Int64')
    
    # Create 'Missing Scope' = max imputed across scopes
    summary_df["Missing Scope Count"] = summary_df[
        ["Imputed Scope 1 Count", "Imputed Scope 2 Count", "Imputed Scope 3 Count"]
    ].max(axis=1)
    
    summary_df["Missing Scope (%)"] = (
        summary_df["Missing Scope Count"] / summary_df["Number of Stocks"] * 100
    ).round(0).astype('Int64')
    
    # Define column order - include filled counts and percentages
    columns_order = ['Period', 'Sector', 'Number of Stocks', 'Missing Scope Count', 'Missing Scope (%)'] + \
                    ['Filled Scope 1 Count', 'Filled Scope 1 %', 
                     'Filled Scope 2 Count', 'Filled Scope 2 %',
                     'Filled Scope 3 Count', 'Filled Scope 3 %'] + \
                    [col for col in summary_df.columns if col not in 
                     ['Period', 'Sector', 'Number of Stocks', 'Missing Scope Count', 'Missing Scope (%)',
                      'Filled Scope 1 Count', 'Filled Scope 1 %',
                      'Filled Scope 2 Count', 'Filled Scope 2 %',
                      'Filled Scope 3 Count', 'Filled Scope 3 %']]
    
    # Only include columns that exist
    columns_order = [col for col in columns_order if col in summary_df.columns]
    
    summary_df = summary_df[columns_order].sort_values("Missing Scope (%)", ascending=False).reset_index(drop=True)
    
    return summary_df

In [4]:
def sector_aggregates(df, period, top_weight_pct=0.2):
    """
    Create sector emissions summary for a single period.
    Shows emission statistics by sector.
    """
    results = []
    
    total_mcap = df["float_mcap"].sum()  # total benchmark market cap
    
    for sector, g in df.groupby("GICS Sector"):
        # Calculate Scope 1+2+3 if not present
        if 'Scope 1+2+3' not in g.columns:
            g['Scope 1+2+3'] = g['Scope 1'] + g['Scope 2'] + g['Scope 3']
        
        # absolute totals
        s1 = g["Scope 1"].sum()
        s2 = g["Scope 2"].sum()
        s3 = g["Scope 3"].sum()
        total_emissions = g["Scope 1+2+3"].sum()
        
        # percentages of each scope
        pct_s1 = s1/total_emissions if total_emissions > 0 else 0
        pct_s2 = s2/total_emissions if total_emissions > 0 else 0
        pct_s3 = s3/total_emissions if total_emissions > 0 else 0
        
        # top 5 emitters in sector
        top5 = g.nlargest(5, "Scope 1+2+3")["Scope 1+2+3"].sum() / total_emissions if total_emissions > 0 else 0
        
        # emissions captured within top X% of sector weights
        g_sorted = g.sort_values("weight_in_sector", ascending=False).copy()
        g_sorted["cum_weight"] = g_sorted["weight_in_sector"].cumsum()
        emissions_top_weight = g_sorted.loc[g_sorted["cum_weight"] <= top_weight_pct, "Scope 1+2+3"].sum()
        pct_emissions_top_weight = emissions_top_weight / total_emissions if total_emissions > 0 else 0
        
        # sector market cap as percentage of benchmark
        sector_mcap = g["float_mcap"].sum()
        sector_mcap_pct = sector_mcap / total_mcap if total_mcap > 0 else 0
        
        results.append({
            "Period": period,
            "Sector": sector,
            "Scope 1 sum": s1,
            "Scope 2 sum": s2,
            "Scope 3 sum": s3,
            "Scope 1+2+3 absolute": total_emissions,
            "% Scope 1": pct_s1,
            "% Scope 2": pct_s2,
            "% Scope 3": pct_s3,
            "% Top 5 emitters": top5,
            f"% Emissions in Top {int(top_weight_pct*100)}% Weight": pct_emissions_top_weight,
            "Avg Scope 1+2+3": g["Scope 1+2+3"].mean(),
            "Avg Scope 1": g["Scope 1"].mean(),
            "Avg Scope 2": g["Scope 2"].mean(),
            "Avg Scope 3": g["Scope 3"].mean(),
            "Var Scope 1": g["Scope 1"].std(),
            "Var Scope 2": g["Scope 2"].std(),
            "Var Scope 3": g["Scope 3"].std(),
            "Avg Carbon Intensity": g["Carbon Intensity"].mean(),
            "Aggregate Carbon Intensity": total_emissions / g["Revenue"].sum() if g["Revenue"].sum() > 0 else 0,
            "Sector Market Cap %": sector_mcap_pct
        })
    
    return pd.DataFrame(results)

In [5]:
def human_format(num):
    """Format large numbers with K, M, B, T suffixes"""
    for unit in ['', 'K', 'M', 'B', 'T']:
        if abs(num) < 1000:
            return f"{num:.1f}{unit}"
        num /= 1000
    return f"{num:.1f}P"

## Load Data for All Periods

In [6]:
# Load benchmark data for all periods
all_periods_data = {}

for period in PERIODS:
    file_path = DATASETS_DIR / f"benchmark_weights_carbon_intensity_{period}.xlsx"
    
    if file_path.exists():
        df = pd.read_excel(file_path)
        print(df.columns)
        # Calculate Scope 1+2+3 if not present
        if 'Scope 1+2+3' not in df.columns:
            df['Scope 1+2+3'] = df['Scope 1'] + df['Scope 2'] + df['Scope 3']
        
        # Calculate float_mcap from weight_in_sector if not present
        if 'float_mcap' not in df.columns:
            # Approximate float_mcap from weights (will be normalized by sector later)
            df['float_mcap'] = df['weight_in_sector']
        
        all_periods_data[period] = df
        print(f"Loaded {period}: {len(df)} companies, {df['GICS Sector'].nunique()} sectors")
    else:
        print(f"Warning: File not found for period {period}")

print(f"\nSuccessfully loaded data for {len(all_periods_data)} periods")

Index(['SYMBOL', 'NAME', 'GICS Sector', 'Scope 1', 'Scope 2', 'Scope 3',
       'Revenue', 'Scope 1 Imputed', 'Scope 2 Imputed', 'Scope 3 Imputed',
       'Filled Scope 1 Count', 'Filled Scope 2 Count', 'Filled Scope 3 Count',
       'Carbon Intensity', 'weight_in_sector', 'TYPE'],
      dtype='object')
Loaded 0321: 497 companies, 11 sectors
Index(['SYMBOL', 'NAME', 'GICS Sector', 'Scope 1', 'Scope 2', 'Scope 3',
       'Revenue', 'Scope 1 Imputed', 'Scope 2 Imputed', 'Scope 3 Imputed',
       'Filled Scope 1 Count', 'Filled Scope 2 Count', 'Filled Scope 3 Count',
       'Carbon Intensity', 'weight_in_sector', 'TYPE'],
      dtype='object')
Loaded 0621: 497 companies, 11 sectors
Index(['SYMBOL', 'NAME', 'GICS Sector', 'Scope 1', 'Scope 2', 'Scope 3',
       'Revenue', 'Scope 1 Imputed', 'Scope 2 Imputed', 'Scope 3 Imputed',
       'Filled Scope 1 Count', 'Filled Scope 2 Count', 'Filled Scope 3 Count',
       'Carbon Intensity', 'weight_in_sector', 'TYPE'],
      dtype='object')
Loaded 

## 1. Sector Imputation Summary

### Individual Period Summaries

In [7]:
# Create imputation summary for each period
imputation_summaries = []

for period, df in all_periods_data.items():
    summary = create_sector_imputation_summary(df, period)
    imputation_summaries.append(summary)
    
    print(f"\n{'='*80}")
    print(f"Imputation Summary - Period {period}")
    print(f"{'='*80}")
    display(summary)


Imputation Summary - Period 0321


,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,0321,Energy,23,11,48,0,0,0,0,2,9,0,0,0,0,11,48
1,0321,Industrials,74,22,30,3,4,3,4,5,7,4,5,4,5,22,30
2,0321,Health Care,63,17,27,4,6,4,6,12,19,4,6,4,6,17,27
3,0321,Financials,74,15,20,4,5,4,5,5,7,4,5,4,5,15,20
4,0321,Consumer Discretionary,58,11,19,1,2,2,3,9,16,2,3,3,5,11,19
5,0321,Information Technology,63,11,17,3,5,3,5,5,8,5,8,5,8,11,17
6,0321,Real Estate,29,5,17,1,3,1,3,4,14,1,3,1,3,5,17
7,0321,Communication Services,22,3,14,2,9,2,9,5,23,3,14,3,14,3,14
8,0321,Consumer Staples,36,4,11,1,3,1,3,0,0,0,0,0,0,4,11
9,0321,Materials,27,2,7,0,0,0,0,5,19,0,0,0,0,2,7



Imputation Summary - Period 0621


,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,0621,Energy,22,10,45,0,0,0,0,2,9,0,0,0,0,10,45
1,0621,Industrials,74,22,30,3,4,3,4,5,7,4,5,4,5,22,30
2,0621,Health Care,63,17,27,4,6,4,6,12,19,4,6,4,6,17,27
3,0621,Financials,74,15,20,4,5,4,5,5,7,4,5,4,5,15,20
4,0621,Consumer Discretionary,58,11,19,1,2,2,3,8,14,2,3,3,5,11,19
5,0621,Information Technology,63,11,17,3,5,3,5,4,6,5,8,5,8,11,17
6,0621,Real Estate,29,5,17,1,3,1,3,4,14,1,3,1,3,5,17
7,0621,Communication Services,22,3,14,1,5,1,5,5,23,3,14,3,14,3,14
8,0621,Consumer Staples,36,4,11,1,3,1,3,1,3,0,0,0,0,4,11
9,0621,Materials,28,2,7,0,0,0,0,5,18,0,0,0,0,2,7



Imputation Summary - Period 0921


...

,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,0623,Energy,23,11,48,2,9,2,9,2,9,2,9,2,9,11,48
1,0623,Health Care,64,20,31,1,2,1,2,4,6,7,11,7,11,20,31
2,0623,Industrials,75,22,29,2,3,2,3,3,4,5,7,5,7,22,29
3,0623,Real Estate,31,8,26,0,0,0,0,0,0,4,13,4,13,8,26
4,0623,Financials,72,16,22,2,3,2,3,3,4,6,8,6,8,16,22
5,0623,Information Technology,65,13,20,3,5,3,5,5,8,8,12,8,12,13,20
6,0623,Consumer Discretionary,52,10,19,3,6,2,4,2,4,2,4,3,6,10,19
7,0623,Consumer Staples,38,6,16,7,18,7,18,7,18,2,5,2,5,6,16
8,0623,Communication Services,20,2,10,1,5,1,5,2,10,2,10,2,10,2,10
9,0623,Materials,29,3,10,1,3,1,3,3,10,1,3,1,3,3,10



Imputation Summary - Period 0923


,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,0923,Energy,23,11,48,2,9,2,9,2,9,2,9,2,9,11,48
1,0923,Health Care,64,20,31,2,3,2,3,4,6,7,11,7,11,20,31
2,0923,Industrials,75,22,29,3,4,3,4,3,4,5,7,5,7,22,29
3,0923,Real Estate,31,8,26,0,0,0,0,0,0,4,13,4,13,8,26
4,0923,Financials,72,17,24,2,3,2,3,3,4,7,10,7,10,17,24
5,0923,Information Technology,65,13,20,9,14,9,14,10,15,8,12,8,12,13,20
6,0923,Consumer Discretionary,52,10,19,3,6,3,6,3,6,3,6,3,6,10,19
7,0923,Consumer Staples,37,5,14,9,24,9,24,9,24,2,5,2,5,5,14
8,0923,Communication Services,20,2,10,2,10,2,10,3,15,2,10,2,10,2,10
9,0923,Materials,29,3,10,1,3,1,3,3,10,1,3,1,3,3,10



Imputation Summary - Period 1223


,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,1223,Energy,23,11,48,2,9,2,9,2,9,2,9,2,9,11,48
1,1223,Industrials,77,25,32,4,5,4,5,5,6,8,10,8,10,25,32
2,1223,Health Care,63,19,30,3,5,3,5,4,6,6,10,6,10,19,30
3,1223,Real Estate,31,8,26,0,0,0,0,0,0,4,13,4,13,8,26
4,1223,Financials,72,17,24,4,6,4,6,4,6,7,10,7,10,17,24
5,1223,Consumer Discretionary,53,11,21,6,11,6,11,5,9,4,8,4,8,11,21
6,1223,Information Technology,64,13,20,15,23,15,23,16,25,8,12,8,12,13,20
7,1223,Consumer Staples,37,5,14,12,32,12,32,11,30,2,5,2,5,5,14
8,1223,Communication Services,19,2,11,2,11,2,11,2,11,2,11,2,11,2,11
9,1223,Materials,28,3,11,3,11,3,11,4,14,1,4,1,4,3,11


### Aggregate Imputation Summary (All Periods)

In [8]:
# Combine all imputation summaries
all_imputation = pd.concat(imputation_summaries, ignore_index=True)

# Build aggregation dict dynamically based on available columns
agg_dict = {
    'Number of Stocks': 'mean',
    'Imputed Scope 1 %': 'mean',
    'Imputed Scope 2 %': 'mean',
    'Imputed Scope 3 %': 'mean'
}

# Add filled percentages if they exist in the data
if 'Filled Scope 1 %' in all_imputation.columns:
    agg_dict['Filled Scope 1 %'] = 'mean'
if 'Filled Scope 2 %' in all_imputation.columns:
    agg_dict['Filled Scope 2 %'] = 'mean'
if 'Filled Scope 3 %' in all_imputation.columns:
    agg_dict['Filled Scope 3 %'] = 'mean'

# Aggregate by sector (averaging across periods)
agg_imputation = all_imputation.groupby('Sector').agg(agg_dict).reset_index()

# Round number of stocks to integer
agg_imputation['Number of Stocks'] = agg_imputation['Number of Stocks'].round(0).astype(int)

# Round percentages to integers
percentage_cols = [col for col in agg_imputation.columns if '%' in col]
agg_imputation[percentage_cols] = agg_imputation[percentage_cols].round(0).astype('Int64')
# Round and clean table
percentage_cols = [col for col in agg_imputation.columns if '%' in col]
agg_imputation[percentage_cols] = agg_imputation[percentage_cols].round(0).astype('Int64')

# Export LaTeX table body only
latex_body = agg_imputation.to_latex(
    index=False,
    escape=False,
    header=False,  # We'll handle headers manually in LaTeX
    float_format="%.0f"
)

with open("results/data_summaries/imputation_summary_body.tex", "w") as f:
    f.write(latex_body)

print("✓ Table body (no header) saved as results/imputation_summary_body.tex")



✓ Table body (no header) saved as results/imputation_summary_body.tex


In [9]:
agg_imputation.columns

Index(['Sector', 'Number of Stocks', 'Imputed Scope 1 %', 'Imputed Scope 2 %',
       'Imputed Scope 3 %', 'Filled Scope 1 %', 'Filled Scope 2 %',
       'Filled Scope 3 %'],
      dtype='object')

## 2. Sector Emissions Summary

### Individual Period Summaries

In [10]:
# Create emissions summary for each period
emissions_summaries = []

for period, df in all_periods_data.items():
    summary = sector_aggregates(df, period)
    emissions_summaries.append(summary)
    
    # Format for display
    summary_display = summary.copy()
    
    pct_cols = ["% Scope 1", "% Scope 2", "% Scope 3", "% Top 5 emitters", 
                "% Emissions in Top 20% Weight", "Sector Market Cap %"]
    summary_display[pct_cols] = summary_display[pct_cols].map(lambda x: f"{x:.1%}")
    
    round_cols = ["Avg Scope 1+2+3", "Avg Scope 1", "Avg Scope 2", "Avg Scope 3",
                  "Var Scope 1", "Var Scope 2", "Var Scope 3",
                  "Avg Carbon Intensity", "Aggregate Carbon Intensity"]
    summary_display[round_cols] = summary_display[round_cols].round(2)
    
    num_cols = ["Scope 1 sum","Scope 2 sum","Scope 3 sum","Scope 1+2+3 absolute"]
    summary_display[num_cols] = summary_display[num_cols].map(human_format)
    
    print(f"\n{'='*80}")
    print(f"Emissions Summary - Period {period}")
    print(f"{'='*80}")
    display(summary_display)


Emissions Summary - Period 0321


,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,0321,Communication Services,4.7M,31.0M,101.0M,136.7M,3.4%,22.7%,73.9%,62.7%,0.0%,"6,213,388.82","212,846.93","1,407,733.10","4,592,808.78","296,163.26","1,862,387.71","6,067,106.86",0.20,0.12,9.1%
1,0321,Consumer Discretionary,36.2M,27.8M,1.3B,1.3B,2.7%,2.1%,95.2%,67.9%,0.0%,"22,840,767.62","624,727.83","478,487.21","21,737,552.57","1,939,659.72","901,982.33","59,062,763.79",1.12,0.78,9.1%
2,0321,Consumer Staples,46.2M,36.1M,1.3B,1.4B,3.3%,2.6%,94.2%,61.7%,13.8%,"39,189,817.92","1,282,626.35","1,002,379.00","36,904,812.57","2,556,044.10","1,684,409.29","60,140,521.27",0.83,0.75,9.1%
3,0321,Energy,342.5M,42.8M,3.3B,3.7B,9.2%,1.2%,89.6%,56.4%,0.0%,"161,246,323.73","14,892,685.57","1,858,785.74","144,494,852.42","22,636,333.69","2,240,399.62","141,657,941.93",5.09,3.50,9.1%
4,0321,Financials,17.8M,10.7M,1.6B,1.6B,1.1%,0.7%,98.2%,94.8%,0.1%,"21,964,245.18","240,866.46","144,655.28","21,578,723.45","1,881,847.26","612,019.14","119,947,032.81",1.83,0.87,9.1%
5,0321,Health Care,33.9M,24.8M,735.8M,794.5M,4.3%,3.1%,92.6%,73.9%,3.9%,"12,611,521.94","538,440.60","393,216.44","11,679,864.89","2,343,206.47","1,196,957.62","38,931,175.07",0.29,0.07,9.1%
6,0321,Industrials,201.9M,17.8M,3.1B,3.3B,6.1%,0.5%,93.3%,64.4%,1.9%,"44,448,922.20","2,728,982.62","240,966.77","41,478,972.81","6,253,934.42","300,367.78","144,957,536.94",3.97,2.43,9.1%
7,0321,Information Technology,31.1M,35.4M,742.2M,808.7M,3.8%,4.4%,91.8%,70.6%,0.0%,"12,837,049.00","493,092.01","562,396.47","11,781,560.52","2,137,141.08","1,230,627.90","42,066,223.41",3.03,0.24,9.1%
8,0321,Materials,144.4M,70.1M,466.9M,681.4M,21.2%,10.3%,68.5%,60.2%,8.7%,"25,237,292.11","5,348,298.74","2,596,797.15","17,292,196.23","7,139,269.01","4,486,483.25","24,771,228.76",1.64,1.59,9.1%
9,0321,Real Estate,8.2M,7.9M,88.9M,105.1M,7.8%,7.6%,84.6%,78.8%,4.5%,"3,624,632.39","284,067.76","273,661.10","3,066,903.53","1,146,888.37","545,873.09","7,428,285.94",0.98,0.82,9.1%



Emissions Summary - Period 0621


,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,0621,Communication Services,3.9M,27.3M,84.6M,115.8M,3.4%,23.6%,73.1%,62.6%,0.0%,"5,265,195.39","178,054.08","1,240,133.67","3,847,007.64","284,907.22","1,868,609.09","4,731,071.57",0.11,0.10,9.1%
1,0621,Consumer Discretionary,36.3M,28.6M,1.3B,1.4B,2.6%,2.0%,95.4%,65.1%,0.0%,"24,337,132.11","625,933.85","492,508.24","23,218,690.02","1,939,288.77","897,581.17","59,405,853.65",1.21,0.82,9.1%
2,0621,Consumer Staples,46.2M,36.1M,1.3B,1.3B,3.4%,2.7%,93.9%,60.5%,14.5%,"37,462,982.00","1,282,626.35","1,002,379.00","35,177,976.65","2,556,044.10","1,684,409.29","58,123,577.24",0.80,0.71,9.1%
3,0621,Energy,337.8M,42.2M,2.9B,3.2B,10.5%,1.3%,88.2%,54.6%,0.0%,"146,914,328.91","15,354,489.45","1,917,548.73","129,642,290.73","23,057,865.38","2,274,907.92","118,744,617.60",4.79,3.10,9.1%
4,0621,Financials,20.6M,13.1M,1.7B,1.7B,1.2%,0.8%,98.1%,96.3%,0.1%,"23,647,010.28","278,210.17","177,642.33","23,191,157.77","1,893,121.00","664,304.88","120,566,523.70",1.85,0.94,9.1%
5,0621,Health Care,15.0M,14.5M,471.2M,500.7M,3.0%,2.9%,94.1%,67.2%,6.3%,"7,948,353.25","237,809.46","230,776.86","7,479,766.93","923,062.12","523,765.34","29,226,948.72",0.22,0.06,9.1%
6,0621,Industrials,207.4M,19.6M,2.8B,3.0B,6.8%,0.6%,92.5%,67.3%,0.9%,"40,993,202.27","2,803,042.64","264,200.48","37,925,959.16","6,271,393.41","334,320.77","144,212,137.98",3.19,2.22,9.1%
7,0621,Information Technology,41.9M,37.6M,740.3M,819.7M,5.1%,4.6%,90.3%,66.8%,0.0%,"13,011,813.40","664,779.82","596,666.52","11,750,367.06","2,851,758.15","1,316,424.25","41,511,624.36",1.88,0.14,9.1%
8,0621,Materials,144.8M,70.7M,506.0M,721.5M,20.1%,9.8%,70.1%,56.8%,8.2%,"25,769,128.43","5,171,573.79","2,525,982.96","18,071,571.68","7,067,949.00","4,418,533.80","24,399,452.10",1.77,1.63,9.1%
9,0621,Real Estate,2.1M,8.6M,110.8M,121.4M,1.7%,7.1%,91.2%,80.3%,3.9%,"4,187,539.89","71,950.41","296,088.66","3,819,500.82","133,766.82","553,452.38","8,906,049.83",1.24,0.94,9.1%



Emissions Summary - Period 0921


...

,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,0623,Communication Services,3.9M,30.3M,95.4M,129.6M,3.0%,23.4%,73.6%,65.0%,0.0%,"6,478,992.73","193,553.60","1,517,265.00","4,768,174.13","268,419.35","2,449,735.78","5,416,660.37",0.12,0.01,9.1%
1,0623,Consumer Discretionary,43.8M,26.5M,1.6B,1.6B,2.7%,1.6%,95.7%,69.2%,0.0%,"31,234,523.03","841,906.31","508,851.38","29,883,765.34","2,443,829.69","915,822.90","75,708,287.59",0.76,0.81,9.1%
2,0623,Consumer Staples,46.2M,36.4M,1.4B,1.4B,3.2%,2.5%,94.3%,59.8%,12.3%,"37,971,479.31","1,216,197.55","957,333.46","35,797,948.30","2,381,876.57","1,671,522.27","60,033,292.73",0.79,0.66,9.1%
3,0623,Energy,335.3M,48.7M,3.4B,3.8B,8.7%,1.3%,90.0%,50.7%,0.0%,"166,693,324.04","14,578,203.02","2,118,823.15","149,996,297.87","21,450,947.52","2,513,059.22","129,882,946.82",6.37,2.95,9.1%
4,0623,Financials,55.9M,12.9M,1.5B,1.5B,3.7%,0.8%,95.5%,91.1%,27.5%,"21,166,848.94","776,679.27","179,674.66","20,210,495.01","6,127,878.35","805,088.64","86,004,263.25",2.42,0.60,9.1%
5,0623,Health Care,10.0M,12.1M,336.2M,358.3M,2.8%,3.4%,93.8%,41.8%,7.4%,"5,598,245.15","155,879.70","188,671.74","5,253,693.71","263,025.66","238,453.17","8,360,102.79",0.42,0.12,9.1%
6,0623,Industrials,234.7M,17.2M,3.2B,3.5B,6.8%,0.5%,92.7%,67.5%,2.5%,"46,326,644.41","3,129,905.00","229,618.16","42,967,121.24","8,164,245.10","271,975.41","150,098,919.18",2.49,2.15,9.1%
7,0623,Information Technology,15.3M,35.9M,306.9M,358.1M,4.3%,10.0%,85.7%,36.5%,0.0%,"5,508,801.14","235,450.53","552,026.32","4,721,324.29","644,962.08","1,242,560.81","7,042,957.27",0.57,0.23,9.1%
8,0623,Materials,141.7M,67.9M,469.2M,678.8M,20.9%,10.0%,69.1%,59.5%,0.0%,"23,406,510.07","4,885,041.96","2,341,129.56","16,180,338.55","6,832,295.82","4,252,559.04","23,140,175.28",1.49,1.44,9.1%
9,0623,Real Estate,5.2M,13.1M,149.7M,168.1M,3.1%,7.8%,89.1%,80.0%,3.2%,"5,421,933.65","168,197.82","423,384.13","4,830,351.70","392,157.25","890,506.23","12,056,970.70",1.63,1.06,9.1%



Emissions Summary - Period 0923


,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,0923,Communication Services,5.0M,30.8M,99.7M,135.5M,3.7%,22.7%,73.6%,63.2%,0.0%,"6,774,174.81","251,583.96","1,537,789.85","4,984,801.00","437,747.06","2,449,682.77","5,287,537.06",0.19,0.02,9.1%
1,0923,Consumer Discretionary,46.7M,27.2M,1.6B,1.7B,2.8%,1.6%,95.6%,67.3%,0.0%,"32,591,985.56","898,753.26","522,459.90","31,170,772.40","2,471,076.05","926,976.63","75,822,712.46",0.86,0.84,9.1%
2,0923,Consumer Staples,66.7M,37.0M,1.4B,1.5B,4.5%,2.5%,93.0%,58.2%,11.9%,"40,084,974.21","1,803,537.29","999,794.56","37,281,642.36","3,438,988.15","1,685,250.73","60,170,913.05",1.00,0.67,9.1%
3,0923,Energy,336.0M,49.1M,3.4B,3.8B,8.8%,1.3%,89.9%,50.3%,0.0%,"165,620,170.21","14,609,890.59","2,134,650.41","148,875,629.22","21,431,106.44","2,505,187.00","127,682,747.09",6.54,2.93,9.1%
4,0923,Financials,16.5M,10.7M,1.7B,1.7B,1.0%,0.6%,98.4%,90.3%,11.2%,"24,064,323.46","228,717.85","149,008.08","23,686,597.53","1,809,375.61","521,663.25","89,457,182.05",2.67,0.69,9.1%
5,0923,Health Care,19.5M,11.7M,321.2M,352.4M,5.5%,3.3%,91.1%,39.5%,7.0%,"5,506,098.16","304,790.37","182,673.27","5,018,634.53","1,164,296.49","219,094.93","7,897,427.39",0.44,0.11,9.1%
6,0923,Industrials,245.5M,17.4M,3.4B,3.7B,6.7%,0.5%,92.8%,64.0%,1.3%,"48,846,524.77","3,273,015.72","232,306.62","45,341,202.43","8,180,582.10","271,317.22","151,500,018.43",2.82,2.27,9.1%
7,0923,Information Technology,28.3M,35.4M,381.2M,444.8M,6.4%,8.0%,85.7%,43.1%,0.0%,"6,843,844.10","434,618.97","544,160.31","5,865,064.83","1,791,038.52","1,236,487.30","11,086,165.80",0.65,0.28,9.1%
8,0923,Materials,141.2M,68.1M,450.9M,660.2M,21.4%,10.3%,68.3%,61.2%,0.0%,"22,765,884.79","4,870,525.34","2,347,143.56","15,548,215.90","6,842,391.42","4,249,283.15","23,249,928.35",1.39,1.40,9.1%
9,0923,Real Estate,9.4M,11.0M,180.0M,200.4M,4.7%,5.5%,89.8%,81.5%,1.8%,"6,465,025.84","302,096.95","356,061.36","5,806,867.54","1,069,006.36","746,273.46","16,167,274.71",1.72,1.27,9.1%



Emissions Summary - Period 1223


,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,1223,Communication Services,3.5M,29.7M,85.6M,118.8M,2.9%,25.0%,72.0%,70.6%,0.0%,"6,251,928.14","182,559.46","1,565,649.21","4,503,719.47","271,167.69","2,507,258.89","5,220,050.10",0.12,0.10,9.1%
1,1223,Consumer Discretionary,44.6M,28.6M,1.5B,1.6B,2.8%,1.8%,95.5%,69.7%,0.0%,"30,354,137.59","840,572.24","539,059.18","28,974,506.16","2,420,992.88","977,735.85","74,896,908.02",0.81,0.79,9.1%
2,1223,Consumer Staples,60.5M,36.7M,1.4B,1.5B,3.9%,2.4%,93.7%,59.2%,11.5%,"41,534,499.02","1,635,932.67","991,786.99","38,906,779.36","3,298,887.23","1,684,560.40","61,520,956.15",0.86,0.69,9.1%
3,1223,Energy,339.9M,53.5M,2.9B,3.3B,10.3%,1.6%,88.1%,52.2%,0.0%,"143,273,545.71","14,776,244.85","2,325,826.11","126,171,474.76","21,331,496.59","2,473,775.33","114,843,798.22",6.16,2.53,9.1%
4,1223,Financials,16.7M,9.2M,1.4B,1.4B,1.2%,0.7%,98.1%,91.8%,22.0%,"19,128,049.49","232,250.58","127,206.21","18,768,592.70","1,758,510.01","385,502.43","82,219,440.48",2.31,0.54,9.1%
5,1223,Health Care,9.1M,11.3M,357.2M,377.5M,2.4%,3.0%,94.6%,44.0%,6.5%,"5,992,283.95","143,928.07","178,836.66","5,669,519.22","223,149.21","228,618.28","9,412,829.55",0.32,0.12,9.1%
6,1223,Industrials,231.4M,23.6M,3.3B,3.5B,6.6%,0.7%,92.8%,66.6%,1.8%,"45,714,118.23","3,005,105.93","307,032.55","42,401,979.75","8,047,615.27","422,287.62","148,199,848.56",2.52,2.13,9.1%
7,1223,Information Technology,11.2M,31.7M,364.5M,407.4M,2.7%,7.8%,89.5%,36.3%,0.0%,"6,365,382.80","174,711.31","496,036.06","5,694,635.43","435,214.32","1,209,701.10","8,703,878.62",0.82,0.25,9.1%
8,1223,Materials,141.3M,68.0M,558.1M,767.4M,18.4%,8.9%,72.7%,64.6%,0.0%,"27,408,745.61","5,046,905.28","2,429,801.40","19,932,038.93","6,900,636.70","4,303,796.57","30,975,787.17",2.00,0.04,9.1%
9,1223,Real Estate,5.3M,10.9M,122.7M,139.0M,3.8%,7.9%,88.3%,71.3%,3.9%,"4,483,136.76","171,451.37","352,701.91","3,958,983.48","436,307.83","750,802.30","7,748,509.90",1.12,0.88,9.1%


### Aggregate Emissions Summary (All Periods)

In [11]:
# Combine all emissions summaries
all_emissions = pd.concat(emissions_summaries, ignore_index=True)

# Aggregate by sector (averaging across periods)
agg_emissions = all_emissions.groupby('Sector').agg({
    'Scope 1 sum': 'mean',
    'Scope 2 sum': 'mean',
    'Scope 3 sum': 'mean',
    'Scope 1+2+3 absolute': 'mean',
    '% Scope 1': 'mean',
    '% Scope 2': 'mean',
    '% Scope 3': 'mean',
    '% Top 5 emitters': 'mean',
    '% Emissions in Top 20% Weight': 'mean',
    'Avg Scope 1+2+3': 'mean',
    'Avg Scope 1': 'mean',
    'Avg Scope 2': 'mean',
    'Avg Scope 3': 'mean',
    'Var Scope 1': 'mean',
    'Var Scope 2': 'mean',
    'Var Scope 3': 'mean',
    'Avg Carbon Intensity': 'mean',
    'Aggregate Carbon Intensity': 'mean',
    'Sector Market Cap %': 'mean'
}).reset_index()

# Load sector volatility data and merge with emissions summary
volatility_file = Path("data/benchmark_returns_volatility/sector_annualized_volatility_2021_2023.xlsx")

if volatility_file.exists():
    volatility_df = pd.read_excel(volatility_file, index_col=0)
    
    # The volatility file has sector names as index and a single column with volatility values
    # Convert to a DataFrame with 'Sector' and 'Annualized Volatility' columns
    volatility_df = volatility_df.reset_index()
    volatility_df.columns = ['Sector', 'Annualized Volatility']
    
    # Merge with aggregate emissions
    agg_emissions = agg_emissions.merge(volatility_df, on='Sector', how='left')
    
    print(f"\n✓ Successfully merged volatility data")
    print(f"  Added 'Annualized Volatility' column to aggregate emissions summary")
else:
    print(f"\n⚠ Volatility file not found at: {volatility_file}")
    print("  Run 'benchmark_replication.py' first to generate volatility data")

# Sort by total emissions
agg_emissions = agg_emissions.sort_values('Scope 1+2+3 absolute', ascending=False).reset_index(drop=True)

# Format for display
agg_emissions_display = agg_emissions.copy()

pct_cols = ["% Scope 1", "% Scope 2", "% Scope 3", "% Top 5 emitters", 
            "% Emissions in Top 20% Weight", "Sector Market Cap %"]
agg_emissions_display[pct_cols] = agg_emissions_display[pct_cols].map(lambda x: f"{x:.1%}")

round_cols = ["Avg Scope 1+2+3", "Avg Scope 1", "Avg Scope 2", "Avg Scope 3",
              "Var Scope 1", "Var Scope 2", "Var Scope 3",
              "Avg Carbon Intensity", "Aggregate Carbon Intensity"]
agg_emissions_display[round_cols] = agg_emissions_display[round_cols].round(2)

# Format Annualized Volatility if it exists
if 'Annualized Volatility' in agg_emissions_display.columns:
    agg_emissions_display['Annualized Volatility'] = agg_emissions_display['Annualized Volatility'].map(lambda x: f"{x:.2%}")

num_cols = ["Scope 1 sum","Scope 2 sum","Scope 3 sum","Scope 1+2+3 absolute"]
agg_emissions_display[num_cols] = agg_emissions_display[num_cols].map(human_format)

print(f"\n{'='*80}")
print(f"AGGREGATE EMISSIONS SUMMARY (Average across {len(PERIODS)} periods)")
print(f"{'='*80}")
display(agg_emissions_display)


✓ Successfully merged volatility data
  Added 'Annualized Volatility' column to aggregate emissions summary

AGGREGATE EMISSIONS SUMMARY (Average across 12 periods)


,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %,Annualized Volatility
0,Energy,335.9M,45.9M,3.2B,3.5B,9.5%,1.3%,89.2%,53.4%,0.0%,"160,125,034.71","15,230,773.70","2,075,910.16","142,818,350.85","22,333,595.06","2,428,939.93","125,625,558.41",5.24,2.80,9.1%,28.47%
1,Industrials,224.8M,19.9M,3.2B,3.5B,6.5%,0.6%,92.9%,65.5%,2.2%,"46,144,384.79","2,997,720.72","265,256.97","42,881,407.10","7,232,918.59","368,431.50","147,304,354.54",3.26,1.72,9.1%,17.25%
2,Consumer Discretionary,41.3M,27.1M,1.4B,1.5B,2.8%,1.8%,95.4%,68.5%,0.0%,"27,708,839.91","763,534.53","499,911.94","26,445,393.44","2,223,718.85","909,064.93","68,970,057.56",0.93,0.80,9.1%,25.68%
3,Consumer Staples,49.5M,36.3M,1.4B,1.4B,3.4%,2.5%,94.1%,60.4%,12.7%,"39,414,163.82","1,347,903.58","989,351.95","37,076,908.29","2,657,223.50","1,686,681.77","60,762,392.55",0.91,0.70,9.1%,13.61%
4,Utilities,642.1M,26.4M,701.9M,1.4B,46.8%,1.9%,51.2%,36.8%,3.3%,"48,254,270.81","22,616,680.48","931,066.03","24,706,524.30","21,674,314.51","1,817,451.49","20,512,449.15",3.87,3.67,9.1%,18.32%
5,Financials,17.9M,11.8M,1.3B,1.3B,1.5%,1.1%,97.4%,91.7%,9.6%,"17,427,049.88","243,765.28","159,484.06","17,023,800.54","1,769,030.23","604,023.20","82,683,764.06",1.81,0.57,9.1%,19.37%
6,Materials,143.0M,69.0M,482.6M,694.5M,20.6%,9.9%,69.4%,59.7%,5.9%,"24,599,752.15","5,063,269.47","2,443,326.38","17,093,156.30","6,957,848.65","4,331,606.92","24,406,882.47",1.51,1.34,9.1%,19.54%
7,Information Technology,31.7M,35.4M,495.3M,562.4M,5.4%,6.9%,87.8%,53.3%,0.0%,"8,879,223.63","501,472.94","557,662.69","7,820,088.01","2,651,350.44","1,281,904.43","24,531,385.69",0.91,0.18,9.1%,25.21%
8,Health Care,17.8M,14.3M,477.6M,509.7M,3.4%,2.9%,93.7%,52.8%,6.4%,"8,076,486.45","281,654.83","227,132.01","7,567,699.60","1,123,448.84","449,636.00","20,424,835.06",0.50,0.13,9.1%,14.27%
9,Communication Services,12.0M,31.8M,166.9M,210.8M,4.9%,18.9%,76.2%,68.5%,0.0%,"9,538,081.63","536,544.97","1,464,621.71","7,536,914.95","1,761,093.20","2,192,334.20","17,303,213.20",0.21,0.07,9.1%,24.08%


In [14]:
volatility_df['Annualized Volatility'].mean()

0.2056809907474607

In [12]:


def compute_sector_structural_metrics(all_periods_data):
    """
    Computes the structural metrics for each sector across periods:
      1a. HHI of benchmark weights
      1b. Largest benchmark weight
      2a. Top 10% carbon intensity weight share
      2b. Weight needed to reach 50% of sector carbon footprint
      2c. Coefficient of variation (CV) of carbon intensity
      4.  Number of constituents

    Returns:
        DataFrame with sectors as rows and metrics averaged across periods.
    """

    results = []

    for period, df in all_periods_data.items():

        # Clean:
        df = df.copy()
        df["Carbon Intensity"] = df["Carbon Intensity"].astype(float)
        df["weight_in_sector"] = df["weight_in_sector"].astype(float)

        # Group by sector
        for sector, sdf in df.groupby("GICS Sector"):

            weights = sdf["weight_in_sector"].values
            carb = sdf["Carbon Intensity"].values

            # -----------------------------
            # 1a. HHI
            # -----------------------------
            hhi = np.sum(weights**2)

            # -----------------------------
            # 1b. Largest constituent weight
            # -----------------------------
            largest_weight = weights.max()

            # -----------------------------
            # 2a. Top 10% most carbon intensive weight share
            # -----------------------------
            n_top = max(1, int(len(sdf) * 0.10))
            sdf_sorted = sdf.sort_values("Carbon Intensity", ascending=False)
            top10_weight = sdf_sorted.head(n_top)["weight_in_sector"].sum()

            # -----------------------------
            # 2b. Weight needed to reach 50% of sector carbon footprint
            # -----------------------------
            sdf_sorted["carbon_contrib"] = sdf_sorted["weight_in_sector"] * sdf_sorted["Carbon Intensity"]
            total_carbon = sdf_sorted["carbon_contrib"].sum()

            sdf_sorted["cum_carbon"] = sdf_sorted["carbon_contrib"].cumsum()
            cutoff_row = sdf_sorted[sdf_sorted["cum_carbon"] >= 0.5 * total_carbon].iloc[0]
            weight_to_half_carbon = sdf_sorted.loc[:cutoff_row.name]["weight_in_sector"].sum()

            # -----------------------------
            # 2c. Coefficient of variation of carbon intensity
            # -----------------------------
            if carb.mean() > 0:
                cv_carbon = carb.std() / carb.mean()
            else:
                cv_carbon = np.nan

            # -----------------------------
            # 4. Number of constituents
            # -----------------------------
            n_const = len(sdf)

            results.append({
                "Period": period,
                "Sector": sector,
                "HHI": hhi,
                "Largest Weight": largest_weight,
                "Top10% CI Weight": top10_weight,
                "Weight to 50% Carbon": weight_to_half_carbon,
                "CV Carbon Intensity": cv_carbon,
                "Num Constituents": n_const
            })

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    # Average across periods
    aggregated = (
        results_df
        .groupby("Sector")
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("Sector")
    )

    return aggregated


In [13]:
metrics_table = compute_sector_structural_metrics(all_periods_data)
print(metrics_table)

vol_2123 = pd.read_excel(
    "data/benchmark_returns_volatility/sector_annualized_volatility_2021_2023.xlsx",
    header=0
)

# Clean: ensure column names are correct
vol_2123.columns = ["Sector", "annual_vol_21_23"]

# Merge on "Sector"
metrics_table = metrics_table.merge(
    vol_2123,
    on="Sector",
    how="left"
)

print(metrics_table)


                    Sector  HHI  Largest Weight  Top10% CI Weight  \
0   Communication Services 0.23            0.42              0.02   
1   Consumer Discretionary 0.14            0.31              0.06   
2         Consumer Staples 0.07            0.14              0.17   
3                   Energy 0.13            0.26              0.03   
4               Financials 0.04            0.09              0.04   
5              Health Care 0.04            0.10              0.04   
6              Industrials 0.03            0.07              0.06   
7   Information Technology 0.15            0.27              0.02   
8                Materials 0.07            0.19              0.06   
9              Real Estate 0.05            0.12              0.07   
10               Utilities 0.07            0.17              0.04   

    Weight to 50% Carbon  CV Carbon Intensity  Num Constituents  
0                   0.35                 1.87             21.75  
1                   0.13               

In [14]:
metrics_table = metrics_table.rename(columns={
    "HHI": "HHI (Benchmark Concentration)",
    "Largest Weight": "Largest Constituent Weight",
    "Top10% CI Weight": "Weight in Top 10% CI Firms",
    "Weight to 50% Carbon": "Weight to 50% Carbon Contribution",
    "CV Carbon Intensity": "CV Carbon Intensity",
    "Num Constituents": "Number of Constituents",
    "annual_vol_21_23": "Annualized Volatility 2021–2023"
})


In [16]:
metrics_table.loc[:, ['Weight in Top 10% CI Firms', 'Weight to 50% Carbon Contribution', 'Annualized Volatility 2021–2023']]

,Weight in Top 10% CI Firms,Weight to 50% Carbon Contribution,Annualized Volatility 2021–2023
0,0.02,0.35,0.24
1,0.06,0.13,0.26
2,0.17,0.26,0.14
3,0.03,0.26,0.28
4,0.04,0.01,0.19
5,0.04,0.10,0.14
6,0.06,0.04,0.17
7,0.02,0.06,0.25
8,0.06,0.35,0.20
9,0.07,0.08,0.20


HHii is more connected to flesibility and sensitivity (inverted), so the higher the concentration less flexibility in theory
and same for large weight and number of constituents

Flexibility is more connected with:
- HHI -> the higher the worse the flexibility
- Largest weight -> the higher the worse the flexibility
- Number of constituents

In [34]:
dri = pd.read_excel("results/DRI/decarbonization_readiness_index.xlsx")

In [36]:
metrics_table = pd.merge(metrics_table, dri, how= 'left', on = 'Sector')

In [37]:
metrics_table.sort_values(by = 'HHI (Benchmark Concentration)', ascending=False)[['Sector', 'HHI (Benchmark Concentration)', 'Flex_norm']] # true but xception for financials that has many stocks

,Sector,HHI (Benchmark Concentration),Flex_norm
0,Communication Services,0.23,0.09
7,Information Technology,0.15,0.25
1,Consumer Discretionary,0.14,0.00
3,Energy,0.13,0.04
8,Materials,0.07,0.24
2,Consumer Staples,0.07,0.19
10,Utilities,0.07,0.41
9,Real Estate,0.05,0.72
5,Health Care,0.04,0.25
4,Financials,0.04,1.00


In [ ]:
metrics_table.sort_values(by = 'Largest Constituent Weight', ascending=False) # true, exception for financials but has many stocks

,Sector,HHI (Benchmark Concentration),Largest Constituent Weight,Weight in Top 10% CI Firms,Weight to 50% Carbon Contribution,CV Carbon Intensity,Number of Constituents,Annualized Volatility 2021–2023
0,Communication Services,0.23,0.42,0.02,0.35,1.87,21.75,0.24
1,Consumer Discretionary,0.14,0.31,0.06,0.13,1.38,54.33,0.26
7,Information Technology,0.15,0.27,0.02,0.06,3.11,63.50,0.25
3,Energy,0.13,0.26,0.03,0.26,0.82,22.08,0.28
8,Materials,0.07,0.19,0.06,0.35,1.25,28.25,0.20
10,Utilities,0.07,0.17,0.04,0.38,0.38,28.42,0.18
2,Consumer Staples,0.07,0.14,0.17,0.26,1.11,36.75,0.14
9,Real Estate,0.05,0.12,0.07,0.08,2.54,30.42,0.20
5,Health Care,0.04,0.10,0.04,0.10,3.12,63.17,0.14
4,Financials,0.04,0.09,0.04,0.01,5.59,73.67,0.19


Room for maneuver is more strongly associated with top 10% CI weight, with weight to 50% carbon contribution and with CV of carbon intensity. So if:
- the top **10% carbon intensity** has a **lower weight** in the portfolio that's **better**
- if the **weight to 50% carbon contribution** is **lower** that's also a **good** thing
- while for the **coefficient of variation of carbon intensity the higher the better** because then there is large spread between lowest and highest carbon intensities, i.e. many opportunities to reweight toward cleaner firms

In [27]:
metrics_table.sort_values(by = 'Weight in Top 10% CI Firms', ascending=True) # doesn't hold

,Sector,HHI (Benchmark Concentration),Largest Constituent Weight,Weight in Top 10% CI Firms,Weight to 50% Carbon Contribution,CV Carbon Intensity,Number of Constituents,Annualized Volatility 2021–2023
0,Communication Services,0.23,0.42,0.02,0.35,1.87,21.75,0.24
7,Information Technology,0.15,0.27,0.02,0.06,3.11,63.50,0.25
3,Energy,0.13,0.26,0.03,0.26,0.82,22.08,0.28
5,Health Care,0.04,0.10,0.04,0.10,3.12,63.17,0.14
4,Financials,0.04,0.09,0.04,0.01,5.59,73.67,0.19
10,Utilities,0.07,0.17,0.04,0.38,0.38,28.42,0.18
8,Materials,0.07,0.19,0.06,0.35,1.25,28.25,0.20
6,Industrials,0.03,0.07,0.06,0.04,2.36,75.00,0.17
1,Consumer Discretionary,0.14,0.31,0.06,0.13,1.38,54.33,0.26
9,Real Estate,0.05,0.12,0.07,0.08,2.54,30.42,0.20


In [28]:
metrics_table.sort_values(by = 'Weight to 50% Carbon Contribution', ascending=True) # doesn't hold

,Sector,HHI (Benchmark Concentration),Largest Constituent Weight,Weight in Top 10% CI Firms,Weight to 50% Carbon Contribution,CV Carbon Intensity,Number of Constituents,Annualized Volatility 2021–2023
4,Financials,0.04,0.09,0.04,0.01,5.59,73.67,0.19
6,Industrials,0.03,0.07,0.06,0.04,2.36,75.00,0.17
7,Information Technology,0.15,0.27,0.02,0.06,3.11,63.50,0.25
9,Real Estate,0.05,0.12,0.07,0.08,2.54,30.42,0.20
5,Health Care,0.04,0.10,0.04,0.10,3.12,63.17,0.14
1,Consumer Discretionary,0.14,0.31,0.06,0.13,1.38,54.33,0.26
3,Energy,0.13,0.26,0.03,0.26,0.82,22.08,0.28
2,Consumer Staples,0.07,0.14,0.17,0.26,1.11,36.75,0.14
0,Communication Services,0.23,0.42,0.02,0.35,1.87,21.75,0.24
8,Materials,0.07,0.19,0.06,0.35,1.25,28.25,0.20


In [30]:
metrics_table.sort_values(by = 'CV Carbon Intensity', ascending=False) # doesn't hold

,Sector,HHI (Benchmark Concentration),Largest Constituent Weight,Weight in Top 10% CI Firms,Weight to 50% Carbon Contribution,CV Carbon Intensity,Number of Constituents,Annualized Volatility 2021–2023
4,Financials,0.04,0.09,0.04,0.01,5.59,73.67,0.19
5,Health Care,0.04,0.10,0.04,0.10,3.12,63.17,0.14
7,Information Technology,0.15,0.27,0.02,0.06,3.11,63.50,0.25
9,Real Estate,0.05,0.12,0.07,0.08,2.54,30.42,0.20
6,Industrials,0.03,0.07,0.06,0.04,2.36,75.00,0.17
0,Communication Services,0.23,0.42,0.02,0.35,1.87,21.75,0.24
1,Consumer Discretionary,0.14,0.31,0.06,0.13,1.38,54.33,0.26
8,Materials,0.07,0.19,0.06,0.35,1.25,28.25,0.20
2,Consumer Staples,0.07,0.14,0.17,0.26,1.11,36.75,0.14
3,Energy,0.13,0.26,0.03,0.26,0.82,22.08,0.28


In [32]:
metrics_table.sort_values(by = 'Annualized Volatility 2021–2023', ascending=False) # doesn't hold

,Sector,HHI (Benchmark Concentration),Largest Constituent Weight,Weight in Top 10% CI Firms,Weight to 50% Carbon Contribution,CV Carbon Intensity,Number of Constituents,Annualized Volatility 2021–2023
3,Energy,0.13,0.26,0.03,0.26,0.82,22.08,0.28
1,Consumer Discretionary,0.14,0.31,0.06,0.13,1.38,54.33,0.26
7,Information Technology,0.15,0.27,0.02,0.06,3.11,63.50,0.25
0,Communication Services,0.23,0.42,0.02,0.35,1.87,21.75,0.24
9,Real Estate,0.05,0.12,0.07,0.08,2.54,30.42,0.20
8,Materials,0.07,0.19,0.06,0.35,1.25,28.25,0.20
4,Financials,0.04,0.09,0.04,0.01,5.59,73.67,0.19
10,Utilities,0.07,0.17,0.04,0.38,0.38,28.42,0.18
6,Industrials,0.03,0.07,0.06,0.04,2.36,75.00,0.17
5,Health Care,0.04,0.10,0.04,0.10,3.12,63.17,0.14


## Save Results

In [12]:
# Create results directory
RESULTS_DIR = Path("results/data_summaries/")
RESULTS_DIR.mkdir(exist_ok=True)

# Save aggregate summaries
output_file = RESULTS_DIR / "sector_summaries_aggregate.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Save aggregate imputation summary
    agg_imputation.to_excel(writer, sheet_name='Imputation Summary', index=False)
    
    # Save aggregate emissions summary (raw numbers, not formatted)
    agg_emissions.to_excel(writer, sheet_name='Emissions Summary', index=False)
    
    # Save all periods imputation data
    all_imputation.to_excel(writer, sheet_name='All Periods Imputation', index=False)
    
    # Save all periods emissions data
    all_emissions.to_excel(writer, sheet_name='All Periods Emissions', index=False)
    
    # Save filled counts by period
    if 'filled_counts_df' in locals():
        filled_counts_df.to_excel(writer, sheet_name='Filled Counts by Period', index=False)

print(f"\nResults saved to: {output_file}")
print(f"  - Imputation Summary (aggregate)")
print(f"  - Emissions Summary (aggregate)")
print(f"  - All Periods Imputation (detailed)")
print(f"  - All Periods Emissions (detailed)")
if 'filled_counts_df' in locals():
    print(f"  - Filled Counts by Period")


Results saved to: results/data_summaries/sector_summaries_aggregate.xlsx
  - Imputation Summary (aggregate)
  - Emissions Summary (aggregate)
  - All Periods Imputation (detailed)
  - All Periods Emissions (detailed)
